# Step 6: Format text changes for manual review
During classification, Gemini can change the input text. 

Make it easy to select which version is better and edit changes.

In [ ]:
from IPython.display import display, HTML
from pathlib import Path
import pandas as pd
data_dir = Path("..") / "data" / "htown_summer26"

ocr_path = data_dir / "04_ocr_output_cleaned.jsonl"
classified_entries_path = data_dir / "05_entries_segmented.csv"
cleaned_entries_path = data_dir / "07_entries_segmented_man_cleaned.csv"
ocr_data_loaded = pd.read_json(ocr_path, lines=True)
classified_entries_loaded = pd.read_csv(classified_entries_path, encoding="utf-8")

In [ ]:
import difflib
def strip_linebreak_diffs(text_in:list[str], text_out:list[str]) -> list[str]:
    """Remove line(entry)breaks from text_in if they don't appear or are spaces in text_out"""
    text1 = "\n".join(text_in)
    text2 = "\n".join(text_out)
    chars_to_ignore_at_EOL = ".,;: -\n\t"
    matcher = difflib.SequenceMatcher(None, text1, text2)
    test_so_far = ""
    for change_type, i1, i2, j1, j2 in matcher.get_opcodes():
        affected_text_in = text1[i1:i2]
        if change_type == 'delete':
            if len(affected_text_in) > 1:
                if affected_text_in.startswith("-\n"):
                    affected_text_in = affected_text_in[2:]
            affected_text_in = affected_text_in.strip()
        if change_type == 'replace':
            if len(affected_text_in) > 1:
                if affected_text_in[0] in chars_to_ignore_at_EOL and affected_text_in[1] == "\n":
                    affected_text_in = affected_text_in[2:]
            if len(affected_text_in) > 1:
                if affected_text_in[-2] in chars_to_ignore_at_EOL and affected_text_in[-1] == "\n":
                    affected_text_in = affected_text_in[:-2]
                elif affected_text_in[-1] in chars_to_ignore_at_EOL:
                    affected_text_in = affected_text_in[:-1]
            changed_to = text2[j1:j2]
            if changed_to.replace(":",";").strip(chars_to_ignore_at_EOL) == affected_text_in.strip(chars_to_ignore_at_EOL):
                test_so_far += changed_to
                continue                
        test_so_far += affected_text_in
        
    return test_so_far.splitlines()

In [ ]:
def compare_texts(row)->bool:
    """Compare two texts and print out the differences with context."""
    text1 = row['ocr_text']
    text2 = row['classified_text']
    print(
        f"⚠ warning: text changed in {row['pub']}.{row['page']}.{row['col']}!"
        f"(length in {len(text1)} vs. out {len(text2)})"
    )
    ignore = lambda x: x in " \n"
    matcher = difflib.SequenceMatcher(ignore, text1, text2)#text2.replace(":",";"))
    texts_differ = False

    for change_type, i1, i2, j1, j2 in matcher.get_opcodes():
        if change_type == 'equal':
            continue
        
        global TEXT_CHANGES
        TEXT_CHANGES += 1
        texts_differ = True

        affected_text_in = text1[i1:i2]
        affected_text_out = text2[j1:j2]

        # Get 10 characters of surrounding context
        context_before_in = text1[max(0, i1 - 10):i1]
        context_after_in = text1[i2:min(len(text1), i2 + 10)]
        context_before_out = text2[max(0, j1 - 10):j1]
        context_after_out = text2[j2:min(len(text2), j2 + 10)]

        # Calculate max length for padding
        len_in = len(affected_text_in)
        len_out = len(affected_text_out)
        max_len = max(len_in, len_out)

        # Pad the strings for consistent display
        if change_type != "delete":
            affected_text_in = affected_text_in#f"{affected_text_in:<{max_len}}"
            affected_text_out = affected_text_out#f"{affected_text_out:<{max_len}}"

        # print(f"  Change Type: {change_type}")
        # # Highlight changes using brackets and padded text
        # print(f"    text from OCR:             '{context_before_in}{{{{{affected_text_in}}}}}{context_after_in}'")
        # print(f"    text after classification: '{context_before_out}{{{{{affected_text_out}}}}}{context_after_out}'")
        # print("--------------------------------------------------")
    return texts_differ

In [ ]:
ocr_data = ocr_data_loaded.copy().rename(columns={"pub": "publication", "page":"page_number", "col":"column", "text": "ocr_text"})
classified_entries = classified_entries_loaded.copy().rename(columns={"full_text": "classified_text"})
merged = ocr_data.drop(columns=["conf", "width", "height"]).merge(
    classified_entries,
    # how="left", #339369 
    #how="inner",  #172845  
    how="outer", # 339391 
    on=["publication", "page_number", "column", "y", "x"],
    validate="one_to_one",
    indicator=True,
    sort=True
)
merged.loc[merged["_merge"] == "left_only", ["width", "height"]] = ocr_data[["width", "height"]]
merged.loc[merged["_merge"] == "left_only", "classified_text"] = ""
merged["width"] = merged["width"].astype(int)
merged["height"] = merged["height"].astype(int)
merged

In [ ]:
# what was on the right and not the left
print(len(merged.loc[merged["_merge"] == "right_only"]))
merged.loc[merged["_merge"] == "right_only"]

In [ ]:
# create last_match to keep track of the last matching row for each non-matching row
merged.loc[merged["_merge"] == "both", "last_match"] = merged[merged["_merge"] == "both"].index
merged["last_match"] = merged.groupby(merged["publication"])["last_match"].ffill()
merged = merged[merged["last_match"].notnull()]
merged[0:20]

In [ ]:
merged["ocr_joined"] = merged.groupby("last_match")["ocr_text"].agg(" ".join)
merged.loc[merged["_merge"] == "left_only", "ocr_joined"] = ""
merged

In [ ]:
from pandas_text_comparer import TextComparer
to_compare = merged[
    merged["classified_text"].str.replace(r"[\s-]", "", regex=True).replace(":",";")
    != 
    merged["ocr_joined"].str.replace(r"[\s-]", "", regex=True)
]
print(len(to_compare))
comparer = TextComparer(to_compare, "ocr_joined", "classified_text")
comparer.run()

In [ ]:
comparer.result

In [ ]:
html = comparer.get_html(to_compare)
html = html.replace(".add {background-color:#aaffaa}", ".add {background-color:#006400}")
html = html.replace(".chg {background-color:#ffff77}", ".chg {background-color:#8B0000}")
html = html.replace(".sub {background-color:#ffaaaa}", ".sub {background-color:#8B8000}")
HTML(html)

In [ ]:
html

In [ ]:
header_text = widgets.HTML(html)
display(header_text)

In [ ]:
merged.iloc[980]["ocr_text"]

In [ ]:

from IPython.display import display, HTML
import ipywidgets as widgets

updates_df = pd.Series()

def update_display(change=None):
    idx = current_idx.value
    row = comparer.result.iloc[idx]
    ocr_display.value = row["ocr_joined"]
    classified_display.value = row["classified_text"]

    index = comparer.result.index[idx]
    current_idx.description = f'Progress ({current_idx.value:{len(str(len(comparer.result)))}d}/{len(comparer.result)}):'

    orig_row = merged.iloc[index]
    header_text.value = f"<h3>Current source column: {orig_row['publication']}, {orig_row['page_number']}, {orig_row['column']}</h3>"
    # print(matched_cols_text.loc[0,"ocr_text"])
    corrected_text.value = to_compare.iloc[idx]["classified_text"]

current_idx = widgets.IntProgress(
    value = 0, 
    min = 0, 
    max = len(comparer.result), 
    step = 1, 
    description = f'Progress ({0:{len(str(len(comparer.result)))}d}/{len(comparer.result)}):',
    bar_style = 'info', 
    orientation = 'horizontal',
    width='900px',
    style={'description_width': 'initial'}
)
current_idx.observe(update_display, names='value')
btn_save_correction = widgets.Button(description='Save & Next', button_style='success')
def save_correction(b):
    idx = current_idx.value
    index = comparer.result.index[idx]
    updates_df.loc[index] = corrected_text.value
    if idx < len(comparer.result) - 1:
        current_idx.value = idx + 1
        current_idx.description = f'Progress ({current_idx.value:{len(str(len(comparer.result)))}d}/{len(comparer.result)}):'
btn_save_correction.on_click(save_correction)

header_text = widgets.HTML('')
controls = widgets.HBox([current_idx, btn_save_correction])

ocr_display = widgets.HTML("", description='OCR text:',)
classified_display = widgets.HTML("",  description='Combined:',)
corrected_text = widgets.Text("", description='Correction:', layout=widgets.Layout(width='70%'))

display(HTML(
    """
<style type='text/css'> 
    .add {background-color:#aaffaa} 
    .chg {background-color:#ffff77} 
    .sub {background-color:#ffaaaa}
</style>
"""
))
display(widgets.VBox([header_text, controls, ocr_display, classified_display, corrected_text]))
update_display()

In [ ]:
updates_df

In [ ]:
cleaned_entries = merged.copy()
cleaned_entries.iloc[updates_df.index, cleaned_entries.columns.get_loc('classified_text')] = updates_df
# cleaned_entries has publication,page_number,column,ocr_text,x,y,entryType,classified_text,width,height,_merge,last_match,ocr_joined
# want publication,page_number,column,entryType,full_text,x,y,width,height
cleaned_entries.rename(columns={"classified_text": "full_text"}, inplace=True)

# remove empty lines
cleaned_entries = cleaned_entries[cleaned_entries['full_text'] != '']

cleaned_entries[
    ["publication","page_number","column","entryType","full_text","x","y","width","height"]
].to_csv(cleaned_entries_path, index=False)
